---
# 🚇 Urban Growth Dynamics of ASEAN Megacities
## Through the Lens of Public Transport Ridership
### การวิเคราะห์การเติบโตของมหานครอาเซียนผ่านเลนส์ระบบขนส่งสาธารณะ
---
## 📋 Executive Summary — บทสรุปผู้บริหาร

&emsp;ลองนึกภาพตามนะคะ — ทุกเช้า คนหลายสิบล้านคนในอาเซียนออกจากบ้าน บางคนขึ้น BTS บางคนนั่ง MRT สิงคโปร์ บางคนต่อรถเมล์ในกัวลาลัมเปอร์ การเคลื่อนไหวของคนเหล่านี้ไม่ใช่แค่ "การเดินทาง" — มันคือ **สัญญาณชีพ** ของเมือง

&emsp;เมื่อเศรษฐกิจดี คนออกไปทำงาน ช้อปปิ้ง พบปะสังสรรค์ — ตัวเลขผู้โดยสารพุ่งขึ้น เมื่อเกิดวิกฤต เช่น COVID-19 — ตัวเลขดิ่งลงในชั่วข้ามคืน นี่คือสิ่งที่รายงานฉบับนี้ต้องการพิสูจน์

| ส่วน | เมือง/หัวข้อ | สิ่งที่จะค้นพบ |
|------|-------------|----------------|
| **Part 1** | กรุงเทพฯ 🇹🇭 | คนกรุงเทพฯ เดินทางด้วยอะไร เพิ่มขึ้นแค่ไหน? |
| **Part 2** | สิงคโปร์ 🇸🇬 | ระบบขนส่งระดับโลกเขาทำได้อย่างไร? |
| **Part 3** | กัวลาลัมเปอร์ 🇲🇾 | มาเลย์กำลังไปในทิศทางไหน? |
| **Part 4** | จาการ์ตา 🇮🇩 | เมืองที่ใหญ่ที่สุดในอาเซียนอยู่ตรงไหน? |
| **Part 5** | มะนิลา 🇵🇭 | ฟิลิปปินส์โตมา 25 ปีอย่างไร? |
| **Part 6** | Economic Growth | ขนส่งกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม? |
| **Part 7** | Development Gap | เมืองกำลังพัฒนาตามสิงคโปร์ทัน? |

---

## 🗂️ ที่มาและความสำคัญ

&emsp;อาเซียนมีประชากรกว่า **650 ล้านคน** กระจายอยู่ใน 10 ประเทศ และกำลังก้าวเข้าสู่ยุคของการขยายตัวอย่างรวดเร็ว เมืองใหญ่ๆ เช่น กรุงเทพฯ จาการ์ตา และมะนิลา ต่างแบกรับประชากรหลักสิบล้านคน พร้อมกับความท้าทายด้านการจราจรและมลพิษที่ตามมา

&emsp;ระบบขนส่งสาธารณะจึงไม่ใช่แค่ "รถไฟฟ้า" หรือ "รถเมล์" — มันคือตัวชี้วัดว่าเมืองนั้น **พัฒนาไปถึงไหน** แล้ว

**ข้อมูลที่ใช้วิเคราะห์:**
- 📊 จำนวนผู้โดยสารรายเดือน จำแนกตามระบบขนส่ง (2019–2025)
- 💰 GDP รายเมือง (Billion USD)
- 👥 จำนวนประชากร (ล้านคน)
- 🌫️ ค่าฝุ่น PM2.5 (μg/m³)

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

---
## Part 1 — กรุงเทพมหานคร 🇹🇭
### โครงสร้างการเดินทางในกรุงเทพฯ เป็นอย่างไร?

&emsp;กรุงเทพฯ มีระบบขนส่งสาธารณะหลายระบบที่ทำงานพร้อมกัน ทั้ง BTS Skytrain, MRT, Airport Rail Link, SRT Red Line (เส้นทางใหม่ที่เพิ่งเปิด), MRT Yellow Line และ MRT Pink Line ซึ่งแต่ละระบบมีผู้ดูแลและเงินลงทุนต่างกัน

&emsp;ข้อมูลที่มีครอบคลุมปี **2021–2025** — ซึ่งเป็นช่วงที่น่าสนใจมาก เพราะเป็นช่วงฟื้นตัวหลัง COVID พอดี และยังเป็นช่วงที่มีการเปิดสายรถไฟใหม่หลายเส้นทางด้วย

---

In [ ]:
# ── Bangkok: prep data ────────────────────────────────────────────────────────
bkk = df[(df['City']=='Bangkok') & (df['Year'].between(2021,2025))].copy()
bkk['Mode_Label'] = bkk['Mode'].map(lambda x: MODE_BKK.get(x,x))

# monthly total
bkk_m = bkk.groupby('Date')['Ridership'].sum().reset_index()

# annual by mode — top 5
bkk_yr = bkk.groupby(['Year','Mode_Label'])['Ridership'].sum().reset_index()
top5_bkk = bkk.groupby('Mode_Label')['Ridership'].sum().nlargest(5).index.tolist()
bkk_top = bkk_yr[bkk_yr['Mode_Label'].isin(top5_bkk)]

# share total
bkk_share = bkk.groupby('Mode_Label')['Ridership'].sum().reset_index().sort_values('Ridership',ascending=False)

# YoY by mode
yoy_list = []
for mode in top5_bkk:
    sub = bkk_yr[bkk_yr['Mode_Label']==mode].sort_values('Year').copy()
    sub['YoY'] = sub['Ridership'].pct_change()*100
    yoy_list.append(sub)
bkk_yoy = pd.concat(yoy_list).dropna(subset=['YoY'])

print('Bangkok data ready ✅')

### ตารางที่ 1.1 — แนวโน้มผู้โดยสารรายเดือน กรุงเทพฯ (2021–2025)

&emsp;กราฟนี้เหมือนการวัดชีพจรของกรุงเทพฯ — แต่ละเดือนบอกเราว่าคนออกไปข้างนอกมากน้อยแค่ไหน ในปี 2021 ตัวเลขยังต่ำมาก เพราะ Lockdown ยังไม่คลาย แต่พอเข้าสู่ปี 2022 เป็นต้นมา เมืองค่อยๆ ฟื้นชีวิตกลับมา

In [ ]:
# ── Chart 1.1: Bangkok Monthly Trend ─────────────────────────────────────────
fig = px.area(
    bkk_m, x='Date', y='Ridership',
    title='<b>ตารางที่ 1.1</b>  กรุงเทพฯ — ผู้โดยสารรายเดือน ทุกระบบรวม (2021–2025)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Date':''},
    color_discrete_sequence=[CITY_COLORS['Bangkok']],
)
fig.update_traces(fillcolor='rgba(255,107,107,0.13)', line_color=CITY_COLORS['Bangkok'], line_width=2.2)
fig.update_layout(**base_layout(420), hovermode='x unified',
                  yaxis=dict(tickformat='.2s',**ax_style()),
                  xaxis=ax_style())
fig.show()

> 📌 **สรุป:** จากช่วงต่ำสุด ~6 ล้านเที่ยว/เดือน ในกลางปี 2021 กรุงเทพฯ ฟื้นตัวขึ้นมาถึง **~50 ล้านเที่ยว/เดือน** ในปี 2025 หรือเพิ่มขึ้นกว่า **8 เท่า** ในเวลาเพียง 4 ปี นอกจากนี้ยังเห็น Seasonal Pattern ชัด — ตัวเลขมักลดลงช่วงพฤษภาคม–สิงหาคม (ฤดูฝน) และสูงในช่วงต้นปี

---

### ตารางที่ 1.2 — สัดส่วนการใช้งานแต่ละระบบขนส่ง กรุงเทพฯ (2021–2025)

&emsp;กรุงเทพฯ มีระบบขนส่งหลายระบบทำงานพร้อมกัน แต่ไม่ใช่ทุกระบบที่ได้รับความนิยมเท่ากัน กราฟวงกลมนี้จะบอกว่า ถ้านับผู้โดยสารทั้งหมดในช่วง 4 ปีที่ผ่านมา แต่ละระบบมีส่วนแบ่งเท่าไหร่

In [ ]:
# ── Chart 1.2: Bangkok Mode Share Donut ──────────────────────────────────────
PIE_COLORS = ['#FF6B6B','#74B9FF','#4ECDC4','#FFB347','#C77DFF','#A8E6CF','#FD79A8']

fig = px.pie(
    bkk_share, names='Mode_Label', values='Ridership',
    title='<b>ตารางที่ 1.2</b>  กรุงเทพฯ — สัดส่วนผู้โดยสารแต่ละระบบ (2021–2025 รวม)',
    hole=0.45, color_discrete_sequence=PIE_COLORS,
)
fig.update_traces(textposition='inside', textinfo='percent+label',
                  insidetextorientation='radial', textfont_size=10,
                  marker=dict(line=dict(color=PAPER_BG, width=2)))
fig.update_layout(**base_layout(460, legend_h=False),
                  legend=dict(bgcolor='rgba(22,27,34,0.9)',bordercolor=GRID_C,
                              borderwidth=1,font_size=11,orientation='h',y=-0.15))
fig.show()

> 📌 **สรุป:** **BTS Skytrain ครองส่วนแบ่งสูงสุดกว่า 50%** ตามด้วย MRT อีกราว 31% ทั้งสองระบบรวมกันคิดเป็นกว่า 80% ของการเดินทางทั้งหมด สายใหม่อย่าง Yellow Line และ Pink Line ยังมีสัดส่วนน้อย แต่กำลังเพิ่มขึ้นเรื่อยๆ สะท้อนว่าคนกรุงเทพฯ รอคอยการขยายระบบ Rail มาโดยตลอด

---

### ตารางที่ 1.3 — ผู้โดยสารแต่ละระบบจำแนกรายปี กรุงเทพฯ

&emsp;กราฟแท่งซ้อนนี้ช่วยให้เห็นทั้ง "ขนาดรวม" และ "โครงสร้างภายใน" ไปพร้อมกัน สังเกตดูว่าในแต่ละปี สัดส่วนของแต่ละสายเปลี่ยนไปอย่างไรบ้าง

In [ ]:
# ── Chart 1.3: Bangkok Stacked Bar by Mode ───────────────────────────────────
fig = px.bar(
    bkk_top, x='Year', y='Ridership', color='Mode_Label',
    barmode='stack',
    title='<b>ตารางที่ 1.3</b>  กรุงเทพฯ — ผู้โดยสารแต่ละระบบขนส่ง รายปี (2021–2025)',
    labels={'Ridership':'ผู้โดยสาร','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=PIE_COLORS,
)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()),
                  bargap=0.25)
fig.show()

> 📌 **สรุป:** ในทุกปี BTS และ MRT เติบโตต่อเนื่อง และเริ่มเห็นการเพิ่มขึ้นของ Yellow Line และ Pink Line ในปี 2023–2024 อย่างชัดเจน แสดงว่าการขยายโครงข่ายรถไฟฟ้าของกรุงเทพฯ กำลังส่งผลจริงต่อพฤติกรรมการเดินทาง

---

### ตารางที่ 1.4 — อัตราการเปลี่ยนแปลง YoY แต่ละระบบ กรุงเทพฯ

&emsp;กราฟนี้ตอบคำถามว่า แต่ละสายรถไฟฟ้า "โตเร็วแค่ไหน" เมื่อเทียบปีต่อปี — บางสายอาจตัวเลขรวมน้อย แต่โตเร็วมาก ซึ่งบ่งบอกถึงศักยภาพในอนาคต

In [ ]:
# ── Chart 1.4: Bangkok YoY by Mode ───────────────────────────────────────────
fig = px.bar(
    bkk_yoy, x='Year', y='YoY', color='Mode_Label',
    barmode='group', text_auto='.0f',
    title='<b>ตารางที่ 1.4</b>  กรุงเทพฯ — อัตราการเปลี่ยนแปลงผู้โดยสาร YoY (%) แต่ละระบบ',
    labels={'YoY':'YoY Change (%)','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=PIE_COLORS,
)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()),
                  bargap=0.15)
fig.update_traces(textfont_size=8, textposition='outside')
fig.show()

> 📌 **สรุป:** ปี 2022 ทุกระบบมี YoY Growth พุ่งสูงมาก (บาง Mode สูงกว่า +100%) เพราะเป็นปีแรกที่เมืองเปิดเต็มที่หลัง COVID ส่วนปี 2023–2024 การเติบโตเริ่มชะลอตัวลงสู่ระดับปกติ ซึ่งเป็นสัญญาณว่าตลาดเริ่มเข้าสู่ภาวะ "ปกติใหม่" แล้ว

---
## Part 2 — สิงคโปร์ 🇸🇬
### ระบบขนส่งระดับโลก — เขาทำได้อย่างไร?

&emsp;สิงคโปร์คือ **มาตรฐานทองคำ** ของระบบขนส่งสาธารณะในอาเซียน ประชากร 5.9 ล้านคน แต่กลับมีผู้โดยสารหลายพันล้านเที่ยวต่อปี หมายความว่าคนสิงคโปร์คนหนึ่ง เดินทางด้วยระบบขนส่งสาธารณะเฉลี่ยกว่า **400 เที่ยวต่อปี** หรือมากกว่าวันเว้นวันตลอดปี

&emsp;ข้อมูลสิงคโปร์ครอบคลุมปี **2019–2024** ซึ่งรวมถึงช่วงก่อน COVID ทำให้เราเห็นภาพการตกและฟื้นตัวได้อย่างสมบูรณ์

---